In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn import preprocessing

In [10]:
scada_10s = pd.read_csv(r"c:\Users\amira\Downloads\testbed_system_1_aggregates\scada_resolved_agg_10s.csv")
scada_16s = pd.read_csv(r"c:\Users\amira\Downloads\testbed_system_1_aggregates\scada_resolved_agg_16s.csv") 
scada_30s = pd.read_csv(r"c:\Users\amira\Downloads\testbed_system_1_aggregates\scada_resolved_agg_30s.csv") 

scada_10s = scada_10s.set_index('timestamp')
scada_16s = scada_16s.set_index('timestamp')
scada_30s = scada_30s.set_index('timestamp')    

scada_10s = scada_10s.resample('30s').mean()   
scada_16s = scada_16s.resample('30s').mean()

merged_scada = scada_10s.join(scada_16s, how= 'outer', lsuffix='_10s', rsuffix='_16s')
merged_scada = merged_scada.join(scada_30s, how= 'outer', lsuffix='_16s', rsuffix='_30s')   
merged_scada = merged_scada.sort_index()

merged_scada = merged_scada.reset_index() 

merged_scada.to.csv(r"c:\Users\amira\Downloads\testbed_system_1_aggregates\merged_scada_30s.csv", index=False)  

print(merged_scada.head())

KeyError: "None of ['timestamp'] are in the columns"

In [ ]:
# Diagnostic: show columns and first rows for each loaded scada file
for name, df in [('scada_10s', scada_10s), ('scada_16s', scada_16s), ('scada_30s', scada_30s)]:
    print(f"--- {name} columns ---")
    print(df.columns.tolist())
    print(df.head(3))
    print()

# Try to infer timestamp-like column names if 'timestamp' not present
possible_ts_names = ['timestamp', 'time', 'datetime', 'ts', 'index']
for name, df in [('scada_10s', scada_10s), ('scada_16s', scada_16s), ('scada_30s', scada_30s)]:
    found = [c for c in df.columns if c.lower() in possible_ts_names or 'time' in c.lower()]
    print(f"{name} - candidate timestamp-like columns: {found}")


In [ ]:
# Robust merge: detect timestamp column, convert to datetime, set index, resample, merge, and save

def prepare_df(df, name, resample_period='30s'):
    # Find timestamp-like column
    ts_candidates = [c for c in df.columns if c.lower() in ('timestamp','time','datetime','ts','index') or 'time' in c.lower()]
    if len(ts_candidates) == 0:
        raise KeyError(f"No timestamp-like column found in {name}; columns: {df.columns.tolist()}")
    ts_col = ts_candidates[0]
    print(f"Using '{ts_col}' as timestamp for {name}")

    # Convert to datetime and set as index
    df[ts_col] = pd.to_datetime(df[ts_col], errors='coerce')
    if df[ts_col].isnull().all():
        raise ValueError(f"All values in {name}.{ts_col} could not be parsed as datetime")
    df = df.set_index(ts_col)

    # Resample and take mean for numeric columns
    df_resampled = df.resample(resample_period).mean()
    return df_resampled

s10 = prepare_df(scada_10s, 'scada_10s', resample_period='30s')
s16 = prepare_df(scada_16s, 'scada_16s', resample_period='30s')
s30 = prepare_df(scada_30s, 'scada_30s', resample_period='30s')

merged_scada = s10.join(s16, how='outer', lsuffix='_10s', rsuffix='_16s')
merged_scada = merged_scada.join(s30, how='outer', rsuffix='_30s')
merged_scada = merged_scada.sort_index().reset_index()

out_path = r"c:\Users\amira\Downloads\testbed_system_1_aggregates\merged_scada_30s.csv"
merged_scada.to_csv(out_path, index=False)
print(f"Saved merged file to {out_path}")
print(merged_scada.head())
